# Column Timing Audit — User Workshop

**This is your audit.** Work from the top, one cell at a time. Read each question, run the evidence cell, then make and record your own bin choice in the next code cell. The notebook does not pre-fill the classifications.

The earlier `docs/COLUMNS.md` contains AI-generated preliminary suggestions. Treat them only as material to review; they are not your decisions. Do not copy a bin without understanding the evidence and reason.

By the end, you should be able to explain when each field is known, whether it can change, and whether it is safe for the intended decision time. The public export alone may not answer that: the source article says data values were extracted from booking/change records as of the day before arrival.


## How to work through the notebook

1. Run each code cell in order, using **Shift+Enter**.
2. Before each decision cell, read the evidence output and the source note in `data/README.md` or the linked paper.
3. In the decision cell, replace `None` with your bin (`1`, `2`, or `3`) and write your reason in your own words.
4. If the source does not settle timing, choose the cautious bin and write what must be verified.
5. The final cell checks completeness and writes `docs/COLUMNS_USER_REVIEW.md` from your recorded choices. It will not write a report while any field is unanswered.

**Bins:** 1 = known at booking and does not change; 2 = amendable or timing uncertain; 3 = after the fact / outcome-related. These definitions follow the project instructions. A field needed to group a hotel-night can still be an operational key; say so in your reason.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display


### Cell 1 — Find and load the project data

Run this cell. It locates the repository from the notebook's current folder and reads the original CSV without changing values or rows. If the file is missing, stop and tell me what the error says before downloading or editing anything.


In [ ]:
possible_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (folder for folder in possible_roots if (folder / "data" / "hotels.csv").exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/hotels.csv above the notebook folder. "
        "Open Jupyter from the project repository or check the data download."
    )

data_path = project_root / "data" / "hotels.csv"
bookings = pd.read_csv(data_path)
print(f"Loaded {len(bookings):,} bookings with {len(bookings.columns)} columns.")
print(f"Data file: {data_path}")


### Cell 2 — Confirm the data shape and field names

Check that the downloaded data matches the brief. This cell reports metadata only; it does not classify fields for you.


In [ ]:
print("Rows:", f"{len(bookings):,}")
print("Columns:", len(bookings.columns))
display(pd.DataFrame({"column_name": bookings.columns, "data_type": bookings.dtypes.astype(str).values}))


### Cell 3 — Build the arrival-date split

The scoring periods are based on arrival date. This cell shows row counts and date limits only. It deliberately does not inspect cancellation outcomes in the proof period.


In [ ]:
arrival_date = pd.to_datetime(
    bookings["arrival_date_year"].astype(str)
    + "-"
    + bookings["arrival_date_month"]
    + "-"
    + bookings["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)

periods = {
    "train": arrival_date < "2016-07-01",
    "choose": (arrival_date >= "2016-07-01") & (arrival_date < "2017-01-01"),
    "proof": (arrival_date >= "2017-01-01") & (arrival_date < "2017-09-01"),
}

period_summary = []
for name, mask in periods.items():
    period_summary.append({
        "period": name,
        "rows": int(mask.sum()),
        "first_arrival": str(arrival_date[mask].min().date()),
        "last_arrival": str(arrival_date[mask].max().date()),
    })
display(pd.DataFrame(period_summary))

# Feature examples below use train + choose rows only. Proof is reserved for final evaluation.
train_choose = bookings.loc[periods["train"] | periods["choose"]].copy()
print(f"Rows available for this audit's value examples: {len(train_choose):,}")


### Cell 4 — Keep the two hotels distinct

Look at the hotel labels and counts. Decide in your own words whether `hotel` is a feature, a grouping key, or both for this project. Do not infer different operating policies just from different cancellation rates.


In [ ]:
hotel_counts = (
    train_choose.groupby("hotel", dropna=False)
    .size()
    .rename("bookings")
    .reset_index()
)
display(hotel_counts)


### Cell 5 — Prepare your decision notebook

The dictionary starts empty. Each group below adds your answers. Do not fill in a bin until you have reviewed that group's evidence and written a reason.


In [ ]:
my_decisions = {}


### Cell group 6 — Group 1 — Property and outcome

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which field identifies the property? Which field is the target/outcome, so it must never be a predictor? Explain the difference.


In [ ]:
show_columns = ['hotel', 'is_canceled']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "hotel": {"bin": None, "reason": ""},
    "is_canceled": {"bin": None, "reason": ""},
})


### Cell group 7 — Group 2 — Booking lead time and arrival calendar

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which values are known when the booking is first made? Can the arrival date change later? Which values are keys needed to group the action by hotel-night?


In [ ]:
show_columns = ['lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "lead_time": {"bin": None, "reason": ""},
    "arrival_date_year": {"bin": None, "reason": ""},
    "arrival_date_month": {"bin": None, "reason": ""},
    "arrival_date_week_number": {"bin": None, "reason": ""},
    "arrival_date_day_of_month": {"bin": None, "reason": ""},
})


### Cell group 8 — Group 3 — Length of stay

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Could a guest amend the stay length? Does the final export prove the original booking-time value?


In [ ]:
show_columns = ['stays_in_weekend_nights', 'stays_in_week_nights']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "stays_in_weekend_nights": {"bin": None, "reason": ""},
    "stays_in_week_nights": {"bin": None, "reason": ""},
})


### Cell group 9 — Group 4 — Party size

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Could the party composition change after booking? What does the source say about changes to guest attributes?


In [ ]:
show_columns = ['adults', 'children', 'babies']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "adults": {"bin": None, "reason": ""},
    "children": {"bin": None, "reason": ""},
    "babies": {"bin": None, "reason": ""},
})


### Cell group 10 — Group 5 — Meal, geography, and sales categories

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** When would the hotel know each value? Does the source say country is always known before check-in? Are the category values proven immutable at booking time?


In [ ]:
show_columns = ['meal', 'country', 'market_segment', 'distribution_channel']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "meal": {"bin": None, "reason": ""},
    "country": {"bin": None, "reason": ""},
    "market_segment": {"bin": None, "reason": ""},
    "distribution_channel": {"bin": None, "reason": ""},
})


### Cell group 11 — Group 6 — Guest history

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Are these based on history strictly before the current booking? What source detail supports your conclusion?


In [ ]:
show_columns = ['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "is_repeated_guest": {"bin": None, "reason": ""},
    "previous_cancellations": {"bin": None, "reason": ""},
    "previous_bookings_not_canceled": {"bin": None, "reason": ""},
})


### Cell group 12 — Group 7 — Room types and changes

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Which room field is requested and which is assigned? Can the request or assignment change? Does the change count include events after booking creation?


In [ ]:
show_columns = ['reserved_room_type', 'assigned_room_type', 'booking_changes']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "reserved_room_type": {"bin": None, "reason": ""},
    "assigned_room_type": {"bin": None, "reason": ""},
    "booking_changes": {"bin": None, "reason": ""},
})


### Cell group 13 — Group 8 — Deposit and booking partner IDs

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Are payment and partner values guaranteed to be known and fixed at booking creation in this export? What would you need to verify?


In [ ]:
show_columns = ['deposit_type', 'agent', 'company']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "deposit_type": {"bin": None, "reason": ""},
    "agent": {"bin": None, "reason": ""},
    "company": {"bin": None, "reason": ""},
})


### Cell group 14 — Group 9 — Wait, customer category, and rate

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Is each value immediately known at booking time, or derived/updated later? What timestamps would settle the question?


In [ ]:
show_columns = ['days_in_waiting_list', 'customer_type', 'adr']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "days_in_waiting_list": {"bin": None, "reason": ""},
    "customer_type": {"bin": None, "reason": ""},
    "adr": {"bin": None, "reason": ""},
})


### Cell group 15 — Group 10 — Requests and final status

Before deciding, consult `data/README.md` and the cited dataset article. The values shown are examples from train + choose only. Values in a CSV do not by themselves tell you when they became known.

**Questions to answer:** Can requests accumulate? Which fields directly expose a cancellation, no-show, checkout, or the date the outcome was set?


In [ ]:
show_columns = ['required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
rows = []
for column in show_columns:
    values = train_choose[column]
    examples = values.dropna().astype(str).drop_duplicates().head(4).tolist()
    rows.append({
        "column": column,
        "data_type": str(values.dtype),
        "missing_train_choose": int(values.isna().sum()),
        "example_values_train_choose": ", ".join(examples),
    })
display(pd.DataFrame(rows))


In [ ]:
# Make your own call after reviewing the evidence above.
# Replace each None with 1, 2, or 3 and write a short reason in your own words.
my_decisions.update({
    "required_car_parking_spaces": {"bin": None, "reason": ""},
    "total_of_special_requests": {"bin": None, "reason": ""},
    "reservation_status": {"bin": None, "reason": ""},
    "reservation_status_date": {"bin": None, "reason": ""},
})


## Final cell — Check your work and save your decisions

Run this only after you have completed every decision cell. It checks that all 32 fields have a valid bin and a reason, then writes a separate review file. It will not fill gaps or choose bins for you.


In [ ]:
expected_columns = list(bookings.columns)
missing_answers = [column for column in expected_columns if column not in my_decisions]
extra_answers = [column for column in my_decisions if column not in expected_columns]
invalid_answers = [
    column for column, answer in my_decisions.items()
    if answer.get("bin") not in {1, 2, 3} or not answer.get("reason", "").strip()
]

if missing_answers or extra_answers or invalid_answers:
    print("Finish or correct your own decisions before saving.")
    print("Missing:", missing_answers)
    print("Unexpected:", extra_answers)
    print("Missing/invalid bin or reason:", invalid_answers)
else:
    bin_names = {1: "Fixed at booking", 2: "Amendable / timing uncertain", 3: "After the fact"}
    report = pd.DataFrame([
        {"column": column, "your_bin": my_decisions[column]["bin"],
         "bin_name": bin_names[my_decisions[column]["bin"]],
         "your_reason": my_decisions[column]["reason"]}
        for column in expected_columns
    ])
    display(report)

    output_path = project_root / "docs" / "COLUMNS_USER_REVIEW.md"
    lines = [
        "# User-Completed Column Timing Audit",
        "",
        "> Decisions below were entered by the learner in `notebooks/01_column_audit_workshop.ipynb`. Review the evidence and unresolved timing caveats before treating a field as an MVP input.",
        "",
        "| Column | Bin | Reason in my words |",
        "|---|---:|---|",
    ]
    for row in report.itertuples(index=False):
        reason = row.your_reason.replace("|", "\\|").replace("\n", " ")
        lines.append(f"| `{row.column}` | {row.your_bin} — {row.bin_name} | {reason} |")
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Saved your review to: {output_path}")
